# Doc-Parser Quickstart
PP-DocLayout-V3 + GLM-OCR Document Parsing Pipeline

This notebook demonstrates the complete two-stage document parsing pipeline:
1. **PP-DocLayout-V3** (server-side): Layout detection with 23 categories
2. **GLM-OCR 0.9B** (server-side): Text/table/formula recognition

All heavy computation runs on the Z.AI MaaS API — no GPU required.

In [ ]:
import sys
sys.path.insert(0, '../src')

import os
import json
from pathlib import Path

# Set your Z.AI API key (or use .env file)
# os.environ["Z_AI_API_KEY"] = "your-key-here"

from doc_parser.config import get_settings, configure_logging
configure_logging("INFO")

try:
    settings = get_settings()
    print(f"\u2713 Settings loaded")
    print(f"  config: {settings.config_yaml_path}")
    print(f"  output: {settings.output_dir}")
except Exception as e:
    print(f"\u26a0 Settings error: {e}")
    print("  Set Z_AI_API_KEY in .env or os.environ to run API calls")

In [ ]:
from doc_parser.pipeline import DocumentParser, ParseResult

# Replace with your document path
sample_path = Path("../data/raw/sample.pdf")  # or sample.png

if not sample_path.exists():
    print(f"\u26a0 Sample file not found at {sample_path}")
    print("  Place a PDF or image in data/raw/ and update the path above")
else:
    parser = DocumentParser()
    result = parser.parse_file(sample_path)

    print(f"\u2713 Parsed: {sample_path.name}")
    print(f"  Pages: {len(result.pages)}")
    print(f"  Total elements: {result.total_elements}")
    for page in result.pages:
        print(f"  Page {page.page_num}: {len(page.elements)} elements")

In [ ]:
from IPython.display import Markdown, display

if 'result' in dir():
    # Use the SDK's pre-assembled full document markdown (highest quality)
    full_markdown = result.full_markdown or "\n\n".join(p.markdown for p in result.pages if p.markdown)
    display(Markdown(full_markdown))
else:
    # Demo: show what markdown output looks like
    demo_md = """
# Sample Document Title

**Abstract:** This is a sample document demonstrating the PP-DocLayout-V3 + GLM-OCR pipeline.

## 1. Introduction

The pipeline detects 23 layout categories and recognizes text, tables, and formulas.

## 2. Mathematical Content

$$
E = mc^2
$$

| Method | F1 Score | Notes |
|--------|----------|-------|
| PP-DocLayout-V3 + GLM-OCR | 94.62 | OmniDocBench V1.5 #1 |
"""
    display(Markdown(demo_md))
    print("(Demo output — parse a real document to see actual results)")

In [ ]:
from doc_parser.chunker import structure_aware_chunking
from collections import Counter

if 'result' in dir():
    all_chunks = []
    for page in result.pages:
        chunks = structure_aware_chunking(
            page.elements,
            source_file=Path(result.source_file).name,
            page=page.page_num,
        )
        all_chunks.extend(chunks)

    modality_counts = Counter(c.modality for c in all_chunks)
    print(f"✓ Generated {len(all_chunks)} RAG-ready chunks")
    print(f"  Modalities: {dict(modality_counts)}")
    print()
    for i, chunk in enumerate(all_chunks[:3]):  # show first 3
        print(f"Chunk {i+1}:")
        print(f"  ID       : {chunk.chunk_id}")
        print(f"  Modality : {chunk.modality}")
        print(f"  Types    : {chunk.element_types}")
        print(f"  Atomic   : {chunk.is_atomic}")
        print(f"  Text     : {chunk.text[:120]}...")
        print()
else:
    print("Demo chunk output:")
    print("  Chunk 1: modality='text',    element_types=['document_title', 'paragraph'], is_atomic=False")
    print("  Chunk 2: modality='table',   element_types=['table'],                       is_atomic=True")
    print("  Chunk 3: modality='image',   element_types=['figure'],                      is_atomic=True")
    print("  Chunk 4: modality='formula', element_types=['formula'],                     is_atomic=True")
    print("  (Parse a real document to see actual chunks)")

In [ ]:
import fitz  # PyMuPDF
from PIL import Image, ImageDraw
from IPython.display import display as ipy_display

# bbox_2d values from the API are normalised to 0-1000 range (NOT 0-1 fractions).
# Correct formula: pixel = bbox_value * image_dimension / 1000

LABEL_COLORS = {
    "document_title": (220, 50, 50),
    "paragraph_title": (30, 100, 220),
    "abstract": (20, 160, 160),
    "paragraph": (40, 160, 40),
    "text": (40, 160, 40),
    "table": (230, 120, 0),
    "formula": (150, 50, 220),
    "inline_formula": (180, 80, 220),
    "figure_caption": (0, 180, 200),
    "code_block": (200, 180, 0),
    "algorithm": (220, 0, 180),
    "image": (100, 100, 100),
}

def visualize_page(result, page_num=0, dpi=150):
    """Render a PDF/image page with bounding boxes overlaid."""
    src = Path(result.source_file)
    if src.suffix.lower() == ".pdf":
        doc = fitz.open(str(src))
        page = doc.load_page(page_num)
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        pix = page.get_pixmap(matrix=mat)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        doc.close()
    else:
        img = Image.open(str(src)).convert("RGB")

    draw = ImageDraw.Draw(img, "RGBA")
    w, h = img.size
    page_result = result.pages[page_num]

    for el in page_result.elements:
        if not el.bbox or len(el.bbox) != 4:
            continue
        color = LABEL_COLORS.get(el.label, (180, 180, 0))
        x1 = int(el.bbox[0] * w / 1000)
        y1 = int(el.bbox[1] * h / 1000)
        x2 = int(el.bbox[2] * w / 1000)
        y2 = int(el.bbox[3] * h / 1000)
        if x2 > x1 and y2 > y1:
            draw.rectangle([x1, y1, x2, y2], fill=(*color, 35), outline=(*color, 220), width=2)
            draw.text((x1 + 3, max(y1 - 16, 0)), el.label.replace("_", " "), fill=(255, 255, 255))

    # Resize to reasonable display width
    display_w = 900
    display_h = int(h * display_w / w)
    ipy_display(img.resize((display_w, display_h), Image.LANCZOS))


if 'result' in dir():
    print(f"Visualizing page 1 of {len(result.pages)}")
    visualize_page(result, page_num=0)
    # To visualize other pages: visualize_page(result, page_num=2)
else:
    print("Parse a document first (run cell 2), then re-run this cell.")

## Phase 2 — Ingest: Embed + Upsert to Qdrant

Embed chunks with dense (OpenAI or Gemini) + sparse (BM25 feature hashing) vectors, then upsert into a Qdrant hybrid collection.

**Prerequisites:**
- Qdrant running locally: `docker-compose up -d qdrant`
- `OPENAI_API_KEY` (or `GEMINI_API_KEY` + `EMBEDDING_PROVIDER=gemini`) set in `.env`

In [ ]:
import asyncio
from openai import AsyncOpenAI
from doc_parser.ingestion.embedder import get_embedder, embed_chunks
from doc_parser.ingestion.image_captioner import enrich_image_chunks
from doc_parser.ingestion.vector_store import QdrantDocumentStore

async def run_ingest(chunks, source_path):
    settings = get_settings()

    # Step 1: GPT-4o captions for image/figure chunks (requires OPENAI_API_KEY)
    if settings.image_caption_enabled:
        openai_client = AsyncOpenAI(
            api_key=settings.openai_api_key.get_secret_value() if settings.openai_api_key else None
        )
        chunks = await enrich_image_chunks(chunks, pdf_path=source_path, client=openai_client)
        image_chunks = [c for c in chunks if c.modality == "image"]
        print(f"✓ Captioned {len(image_chunks)} image chunks")

    # Step 2: Embed — provider set by EMBEDDING_PROVIDER in .env (openai | gemini)
    embedder = get_embedder(settings)
    print(f"  Embedding provider : {settings.embedding_provider}")
    print(f"  Embedding model    : {settings.embedding_model}  ({settings.embedding_dimensions}d)")
    dense, sparse = await embed_chunks(chunks, embedder, settings)
    print(f"✓ Embedded {len(dense)} chunks (dense + sparse)")

    # Step 3: Upsert to Qdrant
    store = QdrantDocumentStore(settings)
    await store.create_collection(overwrite=False)
    upserted = await store.upsert_chunks(chunks, dense, sparse)
    print(f"✓ Upserted {upserted} points → collection '{settings.qdrant_collection_name}'")
    return store

if 'all_chunks' in dir() and all_chunks:
    store = asyncio.run(run_ingest(all_chunks, Path(result.source_file)))
else:
    print("⚠ Run the parse + chunk cells first, then re-run this cell.")
    print("  Requires: running Qdrant (docker-compose up -d qdrant) + API keys in .env")

## Phase 3 — Search: Hybrid Retrieval + Re-ranking

Query the Qdrant collection with dense + sparse RRF fusion, then re-rank results with the configured backend (`RERANKER_BACKEND` in `.env`).

Re-ranker options: `openai` (default, GPT-4o-mini) · `jina` (cloud, multimodal) · `bge` (local, fast) · `qwen` (local VLM)

In [ ]:
from doc_parser.ingestion.embedder import get_embedder
from doc_parser.ingestion.vector_store import QdrantDocumentStore
from doc_parser.retrieval.reranker import get_reranker

# ── Configure your query here ─────────────────────────────────────────────────
QUERY = "What is the main contribution of this paper?"
TOP_K = 20      # candidates retrieved from Qdrant before reranking
TOP_N = 5       # results to keep after reranking
FILTER_MODALITY = None  # None | "text" | "image" | "table" | "formula"
USE_RERANK = True
# ─────────────────────────────────────────────────────────────────────────────

async def run_search(query, top_k=20, top_n=5, filter_modality=None, rerank=True):
    settings = get_settings()
    embedder = get_embedder(settings)
    store = QdrantDocumentStore(settings)

    # Hybrid dense + sparse retrieval (RRF fusion)
    candidates = await store.search(
        query_text=query,
        embedder=embedder,
        settings=settings,
        top_k=top_k,
        filter_modality=filter_modality,
    )
    print(f"✓ Retrieved {len(candidates)} candidates from '{settings.qdrant_collection_name}'")

    if rerank and candidates:
        reranker = get_reranker(settings)
        candidates = await reranker.rerank(query, candidates, top_n=top_n)
        print(f"✓ Re-ranked with '{settings.reranker_backend}' → top {len(candidates)} results")
    else:
        for c in candidates:
            c.setdefault("rerank_score", None)
        candidates = candidates[:top_n]

    print()
    for i, r in enumerate(candidates, 1):
        score = r.get("rerank_score")
        score_str = f"{score:.3f}" if score is not None else "n/a"
        print(f"[{i}] score={score_str}  modality={r.get('modality')}  page={r.get('page')}")
        print(f"    {r.get('text', '')[:200]}")
        if r.get("caption"):
            print(f"    caption: {r['caption'][:120]}")
        print()

    return candidates

if 'store' in dir():
    results = asyncio.run(run_search(
        QUERY, top_k=TOP_K, top_n=TOP_N,
        filter_modality=FILTER_MODALITY, rerank=USE_RERANK,
    ))
else:
    print("⚠ Run the ingest cell first to populate the vector store.")
    print("  Then re-run this cell with your query.")